# Assignment 2 - Gathering data from an API

## Topic 19 - The Data Analytics Process

### Scenario

In Assignment 1 you downloaded a file that somebody had already prepared and published. That works when a suitable file exists. Often it does not. You want three specific countries, not two hundred; last month's figures, not last year's; one indicator, not forty. Downloading a 500 MB file to use 200 rows of it is wasteful, and by the time you have downloaded it the numbers may have changed.

An **API** solves this. You describe what you want, and the service sends back only that.

This assignment has two halves:

- **Part 0 is a complete tutorial.** Every line of code is written for you, working, and explained. You run it and read it. It gathers a year of real daily weather for Accra from the Open-Meteo API, cleans it, saves it, charts it, and documents it.
- **Parts 1 to 12 are your assignment.** You do the same nine steps yourself, with a different API of your choosing, using function skeletons that you complete.

Do not skip Part 0. It is not background reading - it is the worked example the rest of the notebook assumes you have understood.

### What you are practising

| Skill | Where |
|---|---|
| Reading API documentation to find the endpoint you need | Part 0, Part 1 |
| Building a URL safely with `urlencode` | Part 0, Part 2 |
| Sending a request with `urllib` and with `requests` | Part 0, Part 3 |
| Checking status codes and handling failures | Part 0, Part 4 |
| Preserving the raw response before touching it | Part 0, Part 3 |
| Exploring an unfamiliar nested JSON structure | Part 0, Part 5 |
| Flattening JSON into a tidy table | Part 0, Part 6 |
| Cleaning, saving, grouping, and charting | Parts 7 to 9 |
| Pagination and rate limits | Part 10 |
| Submitting work with Git and GitHub | Part 12 |

## What you must submit

A GitHub repository named **`itfy-data-gathering-api`** containing:

1. this notebook, completed, **run from top to bottom so all outputs and charts are visible**;
2. `data/raw/` - the API responses saved exactly as received, as `.json` files;
3. `data/clean/` - the tidy table you built from them;
4. `charts/` - your saved chart images;
5. `manifest.json` - your record of endpoints, parameters, and retrieval times;
6. `README.md` - your research question, source details, findings, and limitations.

Part 12 explains how. If you have never used GitHub, work through [the GitHub submission guide](../topic19d_appendix_submitting_with_github.ipynb) first - it assumes no previous experience and is written for Windows.

### Rules

- **Complete every function skeleton.** Loose code in a cell instead of a completed function loses the marks for that part.
- **Save the raw response before you parse it.** An API response is not a file sitting on a server waiting for you; ask again tomorrow and you may get different numbers. The saved copy is your only evidence.
- **Never put an API key in this notebook.** All three options need no key. If you use a different API that does, read Part 10 first.
- **Charts are limited to line and bar** for the graded parts. Part 11 is optional space for anything else.
- **Explain every result in words.**

## How an API request actually works

An API - Application Programming Interface - is a doorway into a service designed for programs rather than people. The kind you will use here is a **REST API over HTTP**, and it works like this:

1. **You build a URL** describing what you want.
2. **You send an HTTP GET request** to it.
3. **The server replies** with a status code, some headers, and a body.
4. **You check the status code** before trusting the body.
5. **You parse the body**, which is almost always JSON.
6. **You record what you asked for**, so the request can be repeated.

### Reading a URL

```text
https://archive-api.open-meteo.com/v1/archive?latitude=5.6037&longitude=-0.187&start_date=2023-01-01
|-----|  |---------------------------|  |--------|  |--------------------------------------------|
scheme            host                    path                    query string
```

Everything after the `?` is the **query string**: `key=value` pairs joined by `&`. These are the parameters that narrow your request. This is why you can ask for Accra in 2023 rather than the whole world for all time.

### Status codes worth knowing

| Code | Meaning | What you do |
|---|---|---|
| 200 | Success | Parse the body |
| 400 | Bad request - your parameters are wrong | Re-read the documentation |
| 401 / 403 | Not authorised | Your key is missing, wrong, or lacks permission |
| 404 | Not found - that endpoint or resource does not exist | Check the URL for typos |
| 429 | Too many requests - you hit the rate limit | Slow down, cache, try later |
| 500 / 503 | The server has a problem | Wait and retry; it is not your fault |

### The trap that catches everybody once

**A status code of 200 does not mean you got the data you wanted.** Some APIs return 200 with a body that politely explains your request was wrong. The World Bank does this. So does the deprecated version of the REST Countries API. You must check the *contents* of the response, not only its status code. Part 4 makes you do exactly this on purpose.

### JSON, briefly

JSON looks like Python dictionaries and lists, because both describe the same shapes.

| JSON | Python |
|---|---|
| `{"a": 1}` | `dict` |
| `[1, 2, 3]` | `list` |
| `"text"` | `str` |
| `12.5` | `float` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

That last row matters more than it looks. A JSON `null` is a **missing measurement**. It is not zero. `json.load()` turns it into `None`, and pandas turns it into `NaN`. If you replace it with 0 you have invented data.

In [ ]:
# Run this cell as it is. Do not change it.
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlencode
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
import json

import pandas as pd
import matplotlib.pyplot as plt

try:
    import requests
    REQUESTS_AVAILABLE = True
except ImportError:
    REQUESTS_AVAILABLE = False

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"        # saved API responses, never edited
CLEAN_DIR = DATA_DIR / "clean"    # your tidy tables
CHART_DIR = BASE_DIR / "charts"

for folder in (RAW_DIR, CLEAN_DIR, CHART_DIR):
    folder.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["figure.dpi"] = 110

USER_AGENT = "ITFY-Ghana-student-assignment/1.0"
DEFAULT_HEADERS = {"Accept": "application/json", "User-Agent": USER_AGENT}

print("pandas version:     ", pd.__version__)
print("requests available: ", REQUESTS_AVAILABLE)
print("raw responses:      ", RAW_DIR)
print("clean tables:       ", CLEAN_DIR)

---
---
# Part 0 - Tutorial: a complete API pipeline

**Everything in Part 0 is written for you.** Run each cell, read the explanation, and make sure you understand what happened before moving on. Your own assignment repeats these same nine steps.

### The tutorial question

> How did daily temperature and rainfall in Accra vary through 2023, and which months were the hottest and the wettest?

### The API: Open-Meteo Historical Weather

| Item | Detail |
|---|---|
| Endpoint | `https://archive-api.open-meteo.com/v1/archive` |
| Documentation | https://open-meteo.com/en/docs/historical-weather-api |
| API key needed | No |
| Cost | Free for non-commercial use |
| Data source | Reanalysis models combining weather station and satellite observations |
| Terms | https://open-meteo.com/en/terms |

I chose this API for the tutorial because it needs no key, it covers Accra, and its response has an interesting shape that teaches you something you will meet often: **parallel arrays**. Learning to recognise and flatten that shape is a real skill.

### Step 1 - Read the documentation first

This is the step everybody skips and everybody regrets. Before writing code, the documentation tells you the parameters this endpoint accepts:

| Parameter | Meaning | Our value |
|---|---|---|
| `latitude` | Degrees north | `5.6037` (Accra) |
| `longitude` | Degrees east, negative for west | `-0.1870` (Accra) |
| `start_date` | First day wanted, `YYYY-MM-DD` | `2023-01-01` |
| `end_date` | Last day wanted, `YYYY-MM-DD` | `2023-12-31` |
| `daily` | Comma-separated list of daily variables | `temperature_2m_max,temperature_2m_min,precipitation_sum` |
| `timezone` | Timezone for day boundaries | `Africa/Accra` |

Two things the documentation tells us that we could not have guessed:

- `temperature_2m_max` means the highest air temperature measured **2 metres above the ground**, which is the standard height for weather observations. Not the ground temperature, and not a satellite reading.
- The archive is updated with a delay of about five days, so asking for yesterday returns nothing. This is why the tutorial uses a completed year.

Neither fact is in the data. Both change what you can honestly claim about it.

### Step 2 - Build the URL safely

Never build a query string by joining text together. Compare:

```python
# Fragile - breaks the moment a value contains a space, a slash, or an ampersand
url = base + "?timezone=" + "Africa/Accra" + "&daily=" + "temperature_2m_max,temperature_2m_min"

# Correct - urlencode escapes every value properly
url = base + "?" + urlencode({"timezone": "Africa/Accra",
                              "daily": "temperature_2m_max,temperature_2m_min"})
```

`urlencode` converts characters that have a special meaning in a URL into a safe form: a space becomes `%20`, a slash becomes `%2F`, a comma becomes `%2C`. The server decodes them back. Doing this by hand is where an afternoon disappears.

In [ ]:
# TUTORIAL - run this cell.
OPEN_METEO_ENDPOINT = "https://archive-api.open-meteo.com/v1/archive"

# Parameters live in a dictionary, not buried in a string. This makes them easy
# to change, easy to read, and easy to record in the manifest later.
accra_params = {
    "latitude": 5.6037,
    "longitude": -0.1870,
    "start_date": "2023-01-01",
    "end_date": "2023-12-31",
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "Africa/Accra",
}

tutorial_url = f"{OPEN_METEO_ENDPOINT}?{urlencode(accra_params)}"

print("The request URL:")
print(tutorial_url)
print()
print("Notice what urlencode did to the commas and the slash:")
print("  we wrote:   Africa/Accra")
print("  it sent:    Africa%2FAccra")

### Step 3 - Send the request and save the raw response

Two versions follow, doing the same job with the two tools. Read both.

The important habit in this cell is that **we save the raw response to disk before we parse anything**. The reason is simple: this data does not exist as a file anywhere. If Open-Meteo changes its model, retires the endpoint, or is unreachable next week, this saved file is the only proof of what you analysed. That is why `data/raw/` exists.

In [ ]:
# TUTORIAL - run this cell.
TUTORIAL_RAW_PATH = RAW_DIR / "tutorial_open_meteo_accra_2023_raw.json"

# ---- Version 1: urllib.request (standard library) ---------------------------
request = Request(tutorial_url, headers=DEFAULT_HEADERS)
try:
    with urlopen(request, timeout=30) as response:
        status_code = response.status
        content_type = response.headers.get("Content-Type")
        raw_text = response.read().decode("utf-8")
except HTTPError as error:
    # The server answered, but with an error status such as 404 or 500.
    raise RuntimeError(f"The server returned HTTP {error.code} for {tutorial_url}") from error
except URLError as error:
    # We never reached the server: no internet, DNS failure, wrong host name.
    raise RuntimeError(f"Could not reach the server: {error.reason}") from error

print("Status code: ", status_code)
print("Content type:", content_type)
print("Body length: ", f"{len(raw_text):,} characters")

# Save the raw text EXACTLY as received, before parsing it.
TUTORIAL_RAW_PATH.write_text(raw_text, encoding="utf-8")
print("Saved raw response to:", TUTORIAL_RAW_PATH.name)

# Now parse it. json.loads() reads a string; json.load() reads an open file.
payload = json.loads(raw_text)
print("Parsed into a Python", type(payload).__name__)

# ---- Version 2: the same thing with requests --------------------------------
if REQUESTS_AVAILABLE:
    response = requests.get(OPEN_METEO_ENDPOINT, params=accra_params,
                            headers=DEFAULT_HEADERS, timeout=30)
    response.raise_for_status()
    payload_via_requests = response.json()   # parses JSON in one step
    print("\nrequests status:", response.status_code)
    print("requests built this URL for us:", response.url[:90], "...")
    print("Both tools produced the same data:", payload == payload_via_requests)
else:
    print("\nrequests is not installed. Run: pip install requests")

Notice what `requests` did in that second version. We handed it a plain endpoint and a `params` dictionary, and it built the query string itself - no `urlencode` call needed. It also parsed the JSON with `.json()` instead of `json.loads(response.read().decode("utf-8"))`. That is the convenience you are paying for with an extra installed package.

### Step 4 - Explore the structure before assuming anything

You now hold a Python object of unknown shape. Do not guess at it. Look.

The habit to build here is to work downwards one level at a time: what are the top-level keys, what type is each value, and only then what is inside the interesting ones. Printing the entire response is not exploring - a large response floods your screen and teaches you nothing.

In [ ]:
# TUTORIAL - run this cell.
print("Top-level keys and their types:")
for key, value in payload.items():
    if isinstance(value, dict):
        detail = f"dict with {len(value)} keys: {list(value.keys())}"
    elif isinstance(value, list):
        detail = f"list of {len(value)} items"
    else:
        detail = f"{type(value).__name__} = {value}"
    print(f"  {key:26} {detail}")

print("\nMetadata about the location we actually received:")
for key in ("latitude", "longitude", "elevation", "timezone", "timezone_abbreviation"):
    print(f"  {key:24} {payload[key]}")

print("\nThe units, which the documentation alone would not have given us:")
for variable, unit in payload["daily_units"].items():
    print(f"  {variable:22} {unit}")

print("\nInside 'daily' - the first three values of each array:")
for variable, values in payload["daily"].items():
    print(f"  {variable:22} {values[:3]}  ... ({len(values)} values)")

Read that output carefully, because two things in it matter.

**First, the latitude and longitude that came back are not the ones we sent.** We asked for 5.6037, -0.1870 and the response says something slightly different. This is not an error: the model works on a grid, so it returned the nearest grid cell to Accra. If you were comparing two nearby towns you would need to know whether they land in the same cell, and the response is what tells you.

**Second, look at the shape of `daily`.** It is not a list of records. It is a dictionary of **parallel arrays**:

```text
{
  "time":               ["2023-01-01", "2023-01-02", "2023-01-03", ...],
  "temperature_2m_max": [32.0,          31.8,         33.3,         ...],
  "precipitation_sum":  [0.0,           0.0,          0.0,          ...]
}
```

The first item of every array belongs to the first day, the second to the second day, and so on. The connection between a date and its temperature is carried entirely by **position**. Nothing in the data states it.

This shape is common in scientific and time-series APIs, and it is fragile in exactly one way: if you sorted one array without sorting the others, you would silently attach every temperature to the wrong date and produce a chart that looks completely plausible and is completely wrong. Convert it to a table straight away, before doing anything else to it.

### Step 5 - Flatten into a tidy table

Because every array in `daily` has the same length, pandas can turn the whole dictionary into a DataFrame in one line - each key becomes a column. Tidy means one row per observation and one column per variable, and here one observation is one day in Accra.

In [ ]:
# TUTORIAL - run this cell.
# Each key becomes a column; the arrays line up row by row.
accra_raw_table = pd.DataFrame(payload["daily"])

print("Shape:", accra_raw_table.shape, "- 365 days, 4 columns")
print("\nColumn types as they arrived:")
print(accra_raw_table.dtypes)
print("\nFirst three rows:")
print(accra_raw_table.head(3).to_string(index=False))
print("\nMissing values per column:")
print(accra_raw_table.isna().sum().to_string())

Look at the `time` column's type. It arrived as **text**, not as a date, because JSON has no date type - the documentation told us these were ISO 8601 strings and that is exactly what we got.

While it stays text, `.dt` accessors do not work, sorting is alphabetical rather than chronological, and you cannot group by month. Converting it is the first job of cleaning.

### Step 6 - Clean it

Four decisions, each recorded:

1. convert `time` to a real datetime and rename it to `date`;
2. rename the measurement columns to include their units, so nobody has to guess later - `temperature_max_c`, not `temperature_2m_max`;
3. add the derived columns the analysis needs - month number and month name;
4. check the values are physically plausible before trusting them.

That fourth step is the one that beginners leave out. A temperature of 45 °C in Accra or a negative rainfall figure would mean something had gone wrong upstream, and finding that out now is much cheaper than finding out after you have published a chart.

In [ ]:
# TUTORIAL - run this cell.
accra = accra_raw_table.copy()      # never modify the object we parsed

# 1. Text to real dates. Without this, grouping by month is impossible.
accra["date"] = pd.to_datetime(accra["time"])

# 2. Clear names that carry their units.
accra = accra.rename(columns={
    "temperature_2m_max": "temperature_max_c",
    "temperature_2m_min": "temperature_min_c",
    "precipitation_sum": "rainfall_mm",
})

# 3. Derived columns for the monthly analysis.
accra["month"] = accra["date"].dt.month
accra["month_name"] = accra["date"].dt.strftime("%b")
accra["temperature_range_c"] = accra["temperature_max_c"] - accra["temperature_min_c"]

# Drop the original text column now that "date" replaces it, and put the
# columns in a sensible reading order.
accra = accra[["date", "month", "month_name", "temperature_max_c",
               "temperature_min_c", "temperature_range_c", "rainfall_mm"]]
accra = accra.sort_values("date").reset_index(drop=True)

# 4. Plausibility checks. Print the answers rather than assuming them.
print("Date range:      ", accra["date"].min().date(), "to", accra["date"].max().date())
print("Days covered:    ", len(accra), "(2023 had 365 days)")
print("Max temperature: ", accra["temperature_max_c"].min(), "to",
      accra["temperature_max_c"].max(), "degrees C")
print("Rainfall:        ", accra["rainfall_mm"].min(), "to",
      accra["rainfall_mm"].max(), "mm in a day")
print("Any negative rainfall?", bool((accra["rainfall_mm"] < 0).any()),
      "- rainfall cannot be negative, so False is what we want")
print("Any missing values?   ", bool(accra.isna().any().any()))
print("\nCleaned data:")
print(accra.head(3).to_string(index=False))

### Step 7 - Save the cleaned table

Same discipline as Assignment 1. The raw JSON is evidence; this CSV is your interpretation of it. They live in different folders and the raw one is never edited.

In [ ]:
# TUTORIAL - run this cell.
TUTORIAL_CLEAN_PATH = CLEAN_DIR / "tutorial_accra_daily_weather_2023.csv"
accra.to_csv(TUTORIAL_CLEAN_PATH, index=False)

# Read it straight back. This catches an empty or half-written file at once.
check = pd.read_csv(TUTORIAL_CLEAN_PATH, parse_dates=["date"])
print("Saved:      ", TUTORIAL_CLEAN_PATH.name)
print("File size:  ", f"{TUTORIAL_CLEAN_PATH.stat().st_size:,} bytes")
print("Wrote shape:", accra.shape, " read back:", check.shape,
      " match:", accra.shape == check.shape)

### Step 8 - Analyse and chart

The question asked which months were hottest and wettest, so the daily data has to be grouped into months. `groupby` plus `agg` does this in one step.

Note the two different aggregations. Temperature is **averaged** over the month, because a monthly mean temperature is meaningful. Rainfall is **summed**, because the useful figure is how much rain fell in total that month, not the average of the daily amounts. Choosing between mean and sum is a decision about what the number means, not a technical detail - and getting it backwards is a common way to produce a confidently wrong report.

In [ ]:
# TUTORIAL - run this cell.
monthly = accra.groupby(["month", "month_name"], as_index=False).agg(
    days=("date", "count"),
    mean_max_temp_c=("temperature_max_c", "mean"),
    mean_min_temp_c=("temperature_min_c", "mean"),
    total_rainfall_mm=("rainfall_mm", "sum"),
    wet_days=("rainfall_mm", lambda values: (values > 1.0).sum()),
).round(2)

print("Monthly summary for Accra, 2023:")
print(monthly.to_string(index=False))

hottest = monthly.loc[monthly["mean_max_temp_c"].idxmax()]
wettest = monthly.loc[monthly["total_rainfall_mm"].idxmax()]
driest = monthly.loc[monthly["total_rainfall_mm"].idxmin()]

print(f"\nHottest month: {hottest['month_name']}, "
      f"averaging {hottest['mean_max_temp_c']} degrees C at its daily peak")
print(f"Wettest month: {wettest['month_name']}, "
      f"with {wettest['total_rainfall_mm']} mm of rain over {int(wettest['wet_days'])} wet days")
print(f"Driest month:  {driest['month_name']}, "
      f"with only {driest['total_rainfall_mm']} mm")
print(f"Total rainfall in 2023: {accra['rainfall_mm'].sum():.1f} mm")

Now the two charts. The choice between them follows the rule from Assignment 1: **a line for change over time, a bar for comparing categories.**

Daily temperature through the year is a continuous sequence of 365 points, so it is a line. Total rainfall per month is twelve separate quantities being compared, so it is a bar chart.

The titles below state what this data actually showed - Accra's warmest month was December, its coolest August, and its wettest October. Check that your own output agrees. Reanalysis models are occasionally revised, so if your figures differ slightly from these, the honest response is to change the title to match your data rather than to keep a title that no longer fits. A title is a claim, and a claim has to survive being checked.

In [ ]:
# TUTORIAL - run this cell. A line chart, because time is continuous.
fig, ax = plt.subplots()

ax.plot(accra["date"], accra["temperature_max_c"],
        linewidth=1, color="#c0392b", label="Daily maximum")
ax.plot(accra["date"], accra["temperature_min_c"],
        linewidth=1, color="#2980b9", label="Daily minimum")

# A 7-day rolling mean makes the seasonal pattern visible through the daily
# noise. The raw lines stay on the chart so the reader can see both.
ax.plot(accra["date"], accra["temperature_max_c"].rolling(7).mean(),
        linewidth=2.5, color="#7b241c", label="Maximum, 7-day average")

ax.set_title("Accra was warmest in December 2023 and coolest in August")
ax.set_xlabel("Date")
ax.set_ylabel("Air temperature 2 m above ground (degrees C)")
ax.legend(loc="lower left", fontsize=9)
ax.grid(True, alpha=0.3)
fig.text(0.99, -0.03,
         f"Source: Open-Meteo Historical Weather API, retrieved "
         f"{datetime.now(timezone.utc):%d %b %Y}",
         ha="right", fontsize=8, color="grey")
fig.tight_layout()
fig.savefig(CHART_DIR / "tutorial_accra_daily_temperature.png",
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# TUTORIAL - run this cell. A bar chart, because months are separate categories.
fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(monthly["month_name"], monthly["total_rainfall_mm"],
              color="#2980b9", edgecolor="white")

# Label each bar. A reader should never have to estimate a value from a height.
ax.bar_label(bars, fmt="%.0f", fontsize=9, padding=2)

ax.set_title("Accra's rain fell mainly from March to November 2023, peaking in October")
ax.set_xlabel("Month of 2023")
ax.set_ylabel("Total rainfall (mm)")
ax.grid(True, axis="y", alpha=0.3)
ax.set_axisbelow(True)          # grid behind the bars, not over them
fig.text(0.99, -0.03,
         f"Source: Open-Meteo Historical Weather API, retrieved "
         f"{datetime.now(timezone.utc):%d %b %Y}",
         ha="right", fontsize=8, color="grey")
fig.tight_layout()
fig.savefig(CHART_DIR / "tutorial_accra_monthly_rainfall.png",
            dpi=150, bbox_inches="tight")
plt.show()

### Step 9 - Record what you did

The manifest is what makes this repeatable. Anybody holding it can reconstruct the exact request you sent, know when you sent it, and know what you did to the response afterwards.

Notice that the parameters are recorded as data, not as a sentence. "I got 2023 weather for Accra" is a description; the parameter dictionary below is a reproducible instruction.

In [ ]:
# TUTORIAL - run this cell.
tutorial_manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "research_question": ("How did daily temperature and rainfall in Accra vary "
                          "through 2023, and which months were hottest and wettest?"),
    "api": {
        "name": "Open-Meteo Historical Weather API",
        "endpoint": OPEN_METEO_ENDPOINT,
        "documentation": "https://open-meteo.com/en/docs/historical-weather-api",
        "terms": "https://open-meteo.com/en/terms",
        "api_key_required": False,
        "parameters": accra_params,
        "status_code": status_code,
        "retrieved_utc": datetime.now(timezone.utc).isoformat(),
    },
    "response_metadata": {
        "grid_latitude": payload["latitude"],
        "grid_longitude": payload["longitude"],
        "elevation_m": payload["elevation"],
        "timezone": payload["timezone"],
        "units": payload["daily_units"],
    },
    "files": {
        "raw": str(TUTORIAL_RAW_PATH.relative_to(BASE_DIR)),
        "clean": str(TUTORIAL_CLEAN_PATH.relative_to(BASE_DIR)),
    },
    "grain": "one row per calendar day at the model grid cell nearest Accra",
    "cleaning_steps": [
        "Converted the 'time' strings to datetime and renamed the column to 'date'.",
        "Renamed measurement columns to carry their units, e.g. temperature_max_c.",
        "Derived month, month_name, and temperature_range_c.",
        "Checked that rainfall was never negative and that all 365 days were present.",
        "Averaged temperature by month but summed rainfall by month, because a mean "
        "temperature is meaningful whereas the useful rainfall figure is a total.",
    ],
    "known_limitations": [
        "Values are model reanalysis output, not readings from a station in Accra.",
        "The coordinates returned differ slightly from those requested because the "
        "model works on a grid.",
        "One year cannot show a climate trend; it shows one year's weather.",
        "The archive lags real time by roughly five days.",
    ],
    "tools": {"pandas": pd.__version__},
}

TUTORIAL_MANIFEST_PATH = DATA_DIR / "tutorial_manifest.json"
TUTORIAL_MANIFEST_PATH.write_text(json.dumps(tutorial_manifest, indent=2), encoding="utf-8")
print(f"Wrote {TUTORIAL_MANIFEST_PATH.name}\n")
print(json.dumps(tutorial_manifest, indent=2)[:1200], "...")

### Tutorial recap - the nine steps

You have just completed a full API gathering pipeline. These nine steps are what you now repeat with your own API:

| Step | What you did | Why it mattered |
|---|---|---|
| 1 | Read the documentation | Told us the parameters, and that `temperature_2m_max` means 2 m above ground |
| 2 | Built the URL with `urlencode` | Escaped the slash and commas correctly |
| 3 | Sent the request and **saved the raw response** | The response is not a file on a server; the saved copy is the only evidence |
| 4 | Explored the structure | Revealed parallel arrays and a shifted grid coordinate |
| 5 | Flattened to a tidy table | One row per day, position-based links made explicit |
| 6 | Cleaned it | Real dates, units in the column names, plausibility checked |
| 7 | Saved the clean table separately | Raw stays evidence, clean stays reproducible |
| 8 | Grouped and charted | Line for time, bar for categories; mean for temperature, sum for rainfall |
| 9 | Wrote the manifest | The request is now repeatable by somebody else |

### Answer these before continuing

Check your understanding of the tutorial. Write your answers in the cell below.

1. What would have gone wrong if we had sorted the `temperature_2m_max` array without sorting `time` at the same time? Why would the resulting chart still have looked believable?
2. Why did we save the raw JSON to `data/raw/` before parsing it, when we could always request it again?
3. Why is monthly rainfall summed but monthly temperature averaged? What would a chart of *summed* monthly temperature mean?
4. The response returned a latitude slightly different from the one we requested. Why, and when would that difference matter to an analysis?
5. Name one thing about this data that the documentation told us and the numbers alone never could.

### Your tutorial answers

1. *(Write your answer here.)*
2. *(Write your answer here.)*
3. *(Write your answer here.)*
4. *(Write your answer here.)*
5. *(Write your answer here.)*

---
---
# Your assignment starts here

From this point on, the code is yours to write. You will repeat the nine tutorial steps with a different API.

---
## Part 1 - Choose your API and write your question

Three APIs are offered. **Choose one.** All three are free, need no API key, and return JSON. Each has a genuinely different response shape, so each teaches you something the tutorial did not.

### Option A - World Bank Indicators API

| Item | Detail |
|---|---|
| Endpoint pattern | `https://api.worldbank.org/v2/country/{codes}/indicator/{indicator}` |
| Example | `https://api.worldbank.org/v2/country/GHA;KEN;SEN/indicator/SP.DYN.LE00.IN?date=2000:2023&format=json&per_page=500` |
| Documentation | https://datahelpdesk.worldbank.org/knowledgebase/articles/889392 |
| Key needed | No |
| Response shape | A **two-item list**: `[metadata, records]`. The metadata is pagination information; the records are the observations. |
| Parameters | `date` as `2000:2023`, `format=json` (**required**, or you get XML), `per_page`, `page` |
| What one record is | One country, one indicator, one year |
| Country codes | Three-letter ISO codes joined by semicolons, e.g. `GHA;KEN;SEN;NGA` |
| Its challenges | You must pass `format=json`. Records are nested, so `indicator` and `country` are objects containing a `value`. Values are `null` for years with no data. Results are **paginated**. A wrong indicator code returns HTTP 200 with an error message in the body. |

Useful indicator codes:

| Code | Indicator |
|---|---|
| `SP.DYN.LE00.IN` | Life expectancy at birth, years |
| `SP.POP.TOTL` | Population, total |
| `NY.GDP.PCAP.CD` | GDP per capita, current US$ |
| `EG.ELC.ACCS.ZS` | Access to electricity, % of population |
| `SE.PRM.ENRR` | School enrolment, primary, % gross |
| `SH.STA.BASS.ZS` | People using at least basic sanitation services, % |

**Your charts:** a line chart of the indicator over time with one line per country; a bar chart ranking the countries in the most recent year with data.

### Option B - Frankfurter foreign exchange rates

| Item | Detail |
|---|---|
| Endpoint pattern | `https://api.frankfurter.dev/v1/{start}..{end}` |
| Example | `https://api.frankfurter.dev/v1/2023-01-01..2023-12-31?base=USD&symbols=EUR,GBP,JPY,ZAR` |
| Documentation | https://frankfurter.dev |
| Key needed | No |
| Response shape | A dictionary whose `rates` key is a **dictionary keyed by date**, each value being a dictionary of currency to rate. A dictionary of dictionaries, not a list. |
| Parameters | `base` for the reference currency, `symbols` for a comma-separated list of currencies |
| What one record is | One currency's rate against the base on one date |
| Its challenges | The dates are dictionary **keys**, not a column, so flattening needs `orient="index"`. Data covers working days only, so weekends and holidays are simply absent rather than null. The published `start_date` may differ from what you requested, because the API moves back to the last day with data. |
| **Important limit** | Frankfurter publishes European Central Bank reference rates, which cover **30 currencies and do not include the Ghana cedi (GHS)**. Check `https://api.frankfurter.dev/v1/currencies` for the list. Choose currencies from it and say in your limitations that GHS was unavailable. |

**Your charts:** a line chart of the exchange rate over time with one line per currency; a bar chart of each currency's percentage change over the period.

### Option C - USGS Earthquake catalogue

| Item | Detail |
|---|---|
| Endpoint | `https://earthquake.usgs.gov/fdsnws/event/1/query` |
| Example | `https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2023-01-01&endtime=2023-12-31&minmagnitude=5` |
| Documentation | https://earthquake.usgs.gov/fdsnws/event/1/ |
| Key needed | No |
| Response shape | **GeoJSON**: a dictionary whose `features` key is a list of objects, each with nested `properties` and `geometry` sub-objects. |
| Parameters | `format=geojson`, `starttime`, `endtime`, `minmagnitude`, `limit`, `orderby` |
| What one record is | One recorded earthquake |
| Size | Magnitude 5 and above for 2023 is about 1,800 events and 1.3 MB |
| Its challenges | Deeply nested, so `pd.json_normalize()` is the right tool. `properties.time` is a **Unix timestamp in milliseconds**, not a date. `properties.place` is free text like `"146 km WSW of Abepura, Indonesia"`, so extracting a country needs string work. Many fields are null. Coordinates live in `geometry.coordinates` as a list of `[longitude, latitude, depth]` - note that longitude comes first, the opposite of the usual spoken order. |

**Your charts:** a line chart of the number of earthquakes per month through the year; a bar chart of counts by magnitude band or by the ten most affected countries.

### Choosing well

Option A is closest to the course's development-data theme and teaches pagination, which you will meet constantly in real work. Option B has the smallest, cleanest response and the most unusual flattening problem. Option C is the most difficult and the most realistic, because messy free-text fields and epoch timestamps are what most real APIs actually hand you.

Your question must be answerable with a **line chart of change over time** and a **bar chart of a comparison**. Write it before you write any code.

In [ ]:
# The three options are defined for you. Read them, then set CHOSEN_API below.
API_OPTIONS = {
    "A": {
        "name": "World Bank Indicators API",
        "endpoint": "https://api.worldbank.org/v2/country/GHA;KEN;SEN/indicator/SP.DYN.LE00.IN",
        "documentation": "https://datahelpdesk.worldbank.org/knowledgebase/articles/889392",
        "params": {"date": "2000:2023", "format": "json", "per_page": 500},
        "raw_filename": "world_bank_indicator_raw.json",
        "response_shape": "two-item list: [metadata, records]",
        "record_meaning": "one country, one indicator, one year",
        "api_key_required": False,
    },
    "B": {
        "name": "Frankfurter foreign exchange rates",
        "endpoint": "https://api.frankfurter.dev/v1/2023-01-01..2023-12-31",
        "documentation": "https://frankfurter.dev",
        "params": {"base": "USD", "symbols": "EUR,GBP,JPY,ZAR"},
        "raw_filename": "frankfurter_rates_raw.json",
        "response_shape": "dictionary of dictionaries, keyed by date",
        "record_meaning": "one currency rate against the base on one date",
        "api_key_required": False,
    },
    "C": {
        "name": "USGS Earthquake catalogue",
        "endpoint": "https://earthquake.usgs.gov/fdsnws/event/1/query",
        "documentation": "https://earthquake.usgs.gov/fdsnws/event/1/",
        "params": {"format": "geojson", "starttime": "2023-01-01",
                   "endtime": "2023-12-31", "minmagnitude": 5},
        "raw_filename": "usgs_earthquakes_raw.json",
        "response_shape": "GeoJSON: a list of features with nested properties",
        "record_meaning": "one recorded earthquake",
        "api_key_required": False,
    },
}

# TODO 1: Choose your API. Replace None with "A", "B", or "C".
CHOSEN_API = None

# TODO 2: Write your research question as a full sentence ending in a question mark.
RESEARCH_QUESTION = ""

# TODO 3: Adjust the parameters for your own question. Copy the dictionary from
#         your chosen option above and change the values - different countries,
#         a different indicator code, different currencies, a different date
#         range or magnitude. Do not simply reuse the example unchanged: part of
#         the assessment is that you chose your own slice of the data.
MY_PARAMS = {}

# TODO 4: If you chose option A, the country codes and indicator code live in
#         the URL path rather than the parameters, so set your own endpoint
#         here. For options B and C, copy the endpoint from your option, and for
#         option B remember the date range is part of the path.
MY_ENDPOINT = ""

# TODO 5: What do you expect to find, and why? One sentence.
EXPECTATION = ""

# ---- Do not change below this line -------------------------------------------
if CHOSEN_API not in API_OPTIONS:
    raise ValueError('Set CHOSEN_API to "A", "B", or "C" before continuing.')

API = API_OPTIONS[CHOSEN_API]
MY_RAW_PATH = RAW_DIR / API["raw_filename"]

print("API:            ", API["name"])
print("Response shape: ", API["response_shape"])
print("One record is:  ", API["record_meaning"])
print("Documentation:  ", API["documentation"])
print("My endpoint:    ", MY_ENDPOINT or "*** still empty ***")
print("My parameters:  ", MY_PARAMS or "*** still empty ***")
print("My question:    ", RESEARCH_QUESTION or "*** still empty ***")

---
## Part 2 - Build your request URL

Write a small function that assembles an endpoint and a parameter dictionary into a full URL. Keeping this in its own function means every request in your notebook is built the same way, and the manifest can record the parameters separately from the URL.

In [ ]:
def build_api_url(endpoint, params=None):
    """Join an endpoint and a parameter dictionary into a complete URL.

    Parameters
    ----------
    endpoint : str
        The address without a query string.
    params : dict or None
        Query parameters. If empty or None, return the endpoint unchanged
        rather than leaving a bare "?" on the end.

    Returns
    -------
    str
    """
    # TODO 6: Use urlencode() on the parameters and join with "?".
    #         Handle the empty case: build_api_url("https://x.com") must return
    #         "https://x.com", not "https://x.com?".
    raise NotImplementedError("Complete TODO 6, then delete this line.")

In [ ]:
# TODO 7: Build your URL and print it. Then check three things and print each
#         answer, because a URL that looks right and is wrong will waste hours:
#           - does the URL contain every parameter you set in MY_PARAMS?
#           - has urlencode escaped any commas as %2C and any slashes as %2F?
#           - if you chose option A, is "format=json" present? Without it the
#             World Bank returns XML and your JSON parsing will fail with a
#             confusing error about the first character.

# YOUR CODE HERE

---
## Part 3 - Send the request and preserve the raw response

Write the request twice, once with each tool, exactly as the tutorial did. Both functions must save the raw response to disk **before** returning parsed data.

Both functions should:

1. build the URL, or let `requests` build it from `params`;
2. send the request with the headers and a timeout;
3. check the status code, and raise a clear error naming the URL if it failed;
4. save the raw response text to `raw_path` unchanged;
5. return a tuple of `(parsed_payload, request_details)`, where `request_details` is a dictionary recording the URL, the parameters, the status code, the byte count, and the retrieval time.

Returning the details alongside the payload is what lets Part 11 build an honest manifest without you having to remember anything.

In [ ]:
def fetch_json_urllib(endpoint, params=None, raw_path=None, timeout=30):
    """Request JSON with the standard library, saving the raw response first.

    Returns
    -------
    (payload, request_details) : (object, dict)
        payload is the parsed JSON. request_details has the keys "tool", "url",
        "params", "status", "bytes", "retrieved_utc", and "raw_path".
    """
    # TODO 8: Build the URL with your build_api_url() function.

    # TODO 9: Create a Request with DEFAULT_HEADERS, open it with urlopen and
    #         the timeout, record response.status, and read the body. Decode it
    #         with .decode("utf-8") to get text.
    #         Catch HTTPError and URLError separately and raise a RuntimeError
    #         that names the URL and explains which of the two happened. The
    #         difference matters when you are debugging: HTTPError means the
    #         server answered and refused, URLError means you never reached it.

    # TODO 10: If raw_path was given, save the raw text there with
    #          .write_text(raw_text, encoding="utf-8") BEFORE parsing it.

    # TODO 11: Parse the text with json.loads() and return it together with the
    #          request_details dictionary described above.

    raise NotImplementedError("Complete TODOs 8 to 11, then delete this line.")


def fetch_json_requests(endpoint, params=None, raw_path=None, timeout=30):
    """Request JSON with the requests library, saving the raw response first.

    Returns the same tuple shape as fetch_json_urllib.
    """
    if not REQUESTS_AVAILABLE:
        raise RuntimeError("requests is not installed. Run: pip install requests")

    # TODO 12: Call requests.get() passing params= directly, so requests builds
    #          the query string itself. Include the headers and the timeout.

    # TODO 13: Call response.raise_for_status(), save response.text to raw_path
    #          if given, then return response.json() with the same
    #          request_details dictionary shape, using response.url and
    #          response.status_code.

    raise NotImplementedError("Complete TODOs 12 and 13, then delete this line.")

In [ ]:
# TODO 14: Fetch your data with fetch_json_urllib(), saving the raw response to
#          MY_RAW_PATH. Store the results as payload and request_details.
# TODO 15: Print the request_details, and confirm the raw file now exists on
#          disk and is larger than zero bytes.
# TODO 16: Fetch the same data with fetch_json_requests() into a different raw
#          filename, and check that both tools returned identical payloads with
#          a simple == comparison. If they differ, something is wrong with one
#          of your functions - investigate before continuing.

# YOUR CODE HERE

---
## Part 4 - Make it fail on purpose

Everything so far assumed success. Real gathering code spends most of its life dealing with failure, and you cannot handle an error you have never seen.

You are going to break your request in three different ways and record what each one does. This is a graded part of the assignment, not an optional curiosity.

### The three failures

| Failure | How to cause it | What you should observe |
|---|---|---|
| A wrong host | Change the domain to something that does not exist, e.g. `https://api.worldbank.invalid/v2/...` | No HTTP status at all - you never reached a server. `urllib` raises `URLError`. |
| A wrong path | Keep the host, break the path, e.g. add `/nonsense` to the end | An HTTP error status, usually 404. `urllib` raises `HTTPError`. |
| A wrong parameter | Ask for something that does not exist - a made-up indicator code such as `NOT.A.REAL.CODE`, a currency such as `XYZ`, or a `starttime` after your `endtime` | This is the interesting one. Read on. |

That third case is the trap described at the top of this notebook. Some APIs return **HTTP 200 with a body that explains your request was wrong**. The World Bank does exactly this: ask for a nonexistent indicator and you get a perfectly valid JSON response, status 200, whose content is an error message. Your code, seeing 200, would carry on and then fail somewhere much later with a confusing message about a missing key.

The lesson is: **check the shape of the response, not only its status code.**

In [ ]:
# TODO 17: Cause failure 1 (a wrong host). Wrap the call in try/except, catch
#          the exception, and print the exception TYPE and MESSAGE. Do not let
#          it stop the notebook.
# TODO 18: Cause failure 2 (a wrong path). Do the same.
# TODO 19: Cause failure 3 (a wrong parameter). Do NOT wrap this one in
#          try/except at first - instead print the status code and then print
#          the response body. Look at what you actually received.

# YOUR CODE HERE

In [ ]:
def check_api_payload(payload, api_option):
    """Raise a clear error if a parsed payload is not the data we asked for.

    A status code of 200 is not proof of success. This function inspects the
    parsed body and confirms it has the shape the API is documented to return.

    Parameters
    ----------
    payload : object
        The parsed JSON.
    api_option : str
        "A", "B", or "C".

    Returns
    -------
    object
        The payload unchanged, so this function can be used inline:
            records = check_api_payload(payload, CHOSEN_API)

    Raises
    ------
    ValueError
        With a message explaining exactly what was wrong.
    """
    # TODO 20: Write the check for your chosen API. Only your own option has to
    #          work, but read all three descriptions - recognising these shapes
    #          is the skill.
    #
    #   Option A, World Bank. A good response is a list of exactly 2 items:
    #     metadata dict, then a list of records. An ERROR response is a list
    #     whose first item is a dict containing the key "message". So: raise if
    #     the payload is not a list, if it has fewer than 2 items, or if
    #     payload[0] contains "message". Also raise if payload[1] is empty,
    #     which means your query was valid but matched no data at all.
    #
    #   Option B, Frankfurter. A good response is a dict containing the keys
    #     "base" and "rates", with "rates" not empty. An error response
    #     contains "message" instead.
    #
    #   Option C, USGS. A good response is a dict whose "type" is
    #     "FeatureCollection" and which has a non-empty "features" list.
    #     Also check payload["metadata"]["status"] equals 200 if it is present.

    raise NotImplementedError("Complete TODO 20, then delete this line.")

In [ ]:
# TODO 21: Prove your check works, both ways. This matters more than it looks:
#          a validator that never rejects anything gives false confidence.
#            - Call check_api_payload() on your GOOD payload. It should return
#              the payload without complaint.
#            - Call it on the BAD payload from failure 3, inside a try/except,
#              and print the ValueError message it raises.

# YOUR CODE HERE

### Your answer - what failure looks like

| Failure | Exception type or status code | The message you saw | How you would handle it in production code |
|---|---|---|---|
| Wrong host | [your answer] | [your answer] | [your answer] |
| Wrong path | [your answer] | [your answer] | [your answer] |
| Wrong parameter | [your answer] | [your answer] | [your answer] |

Then answer these:

1. Which of the three failures would be hardest to notice in a script that runs automatically every night, and why? [your answer]
2. Your `check_api_payload()` function rejected the bad payload. Where exactly would your code have failed if you had not written that check - and would the error message have pointed you at the real problem? [your answer]
3. A status code of 200 means the HTTP request succeeded. What does it *not* mean? [your answer]

---
## Part 5 - Explore the structure before assuming it

You have a payload of a shape you have read about but not yet seen. Look at it properly, the way the tutorial did in step 4: one level at a time.

In [ ]:
def describe_json_structure(obj, name="payload", max_depth=2, _depth=0):
    """Print a readable summary of a nested JSON structure without flooding the screen.

    For a dict, print each key with the type of its value, and recurse while
    _depth is below max_depth. For a list, print its length and describe only
    its FIRST item - a list of 1,800 earthquakes has 1,800 items of the same
    shape, and printing them all tells you nothing that the first one does not.
    For anything else, print the type and a short preview of the value.

    Indent each level so the nesting is visible.
    """
    # TODO 22: Implement this. Use isinstance(obj, dict) and
    #          isinstance(obj, list) to decide what to do.
    #          Indent with "  " * _depth, and recurse with _depth + 1.
    #          Truncate long values so a single string cannot fill the screen.
    raise NotImplementedError("Complete TODO 22, then delete this line.")

In [ ]:
# TODO 23: Run describe_json_structure() on your payload and read the output.
# TODO 24: Print, with a label for each:
#            - how many records or observations the response contains
#            - the keys available on a single record
#            - one complete example record, formatted readably with
#              json.dumps(record, indent=2)
# TODO 25: Find and print any metadata the response carries that is NOT the
#          data itself - pagination counts, units, a licence, a generation
#          timestamp, the last-updated date. Every option has some. This is the
#          information that later lets you say how complete your data is.

# YOUR CODE HERE

### Your answer - the shape of your response

1. Draw the structure of your response in text, the way the tutorial drew the parallel arrays. Show the nesting, not every value. [your answer]
2. How many records did you receive? Is that all of them, or only the first page? How do you know? [your answer]
3. Which fields are nested inside another object, so that a single `pd.DataFrame(...)` call would leave you with dictionaries sitting inside cells? [your answer]
4. Which metadata field would you need to quote if somebody challenged how current your data is? [your answer]

---
## Part 6 - Flatten into a tidy table

Now turn the nested response into a table with one row per observation and one column per variable.

The right tool depends on the shape, and there is no single answer - which is exactly why you had to look at the structure first.

| Shape | Tool | Note |
|---|---|---|
| A list of flat dictionaries | `pd.DataFrame(records)` | The simple case |
| A list of **nested** dictionaries | `pd.json_normalize(records)` | Creates columns like `properties.mag` |
| A dictionary of parallel arrays | `pd.DataFrame(the_dict)` | The tutorial's shape |
| A dictionary **keyed by** something meaningful | `pd.DataFrame.from_dict(d, orient="index")` | The key becomes the index, so `.reset_index()` afterwards to make it a column |

### Guidance for your option

**Option A, World Bank.** The records are at `payload[1]`. Each one nests `indicator` and `country` as objects, so use `pd.json_normalize(payload[1])` and you will get columns named `indicator.value` and `country.value`. Rename them to something readable. `date` is the year and arrives as a string. `value` is your measurement and is `None` for years with no data.

**Option B, Frankfurter.** The rates are at `payload["rates"]`, a dictionary keyed by date. Use `pd.DataFrame.from_dict(payload["rates"], orient="index")`, which gives you one row per date and one column per currency, with the dates as the index. Then `.reset_index()` and rename that column to `date`. This leaves you in **wide** format, one column per currency. For a line chart with one line per currency, reshape it to long format with `.melt()` so you have `date`, `currency`, and `rate` columns - this is the tidy shape, and it is worth doing.

**Option C, USGS.** The events are at `payload["features"]`, deeply nested. Use `pd.json_normalize(payload["features"])`, which flattens `properties` and `geometry` into dotted column names. Then keep only the columns you need rather than all thirty. `properties.time` is **milliseconds since 1 January 1970**, so convert it with `pd.to_datetime(column, unit="ms")` - forgetting `unit="ms"` gives you dates in 1970 and is a mistake worth making once, deliberately, so that you recognise it later.

In [ ]:
def flatten_response(payload, api_option):
    """Turn a nested API payload into a tidy DataFrame.

    One row per observation, one column per variable, no dictionaries or lists
    left inside any cell.

    Returns
    -------
    pd.DataFrame
    """
    # TODO 26: Implement the flattening for your chosen option, following the
    #          guidance above. Return the DataFrame - do not print it and
    #          return None, or Part 7 will have nothing to work with.
    raise NotImplementedError("Complete TODO 26, then delete this line.")

In [ ]:
# TODO 27: Flatten your payload into flat_df and print its shape and columns.
# TODO 28: Print the first five rows and the dtypes.
# TODO 29: Confirm nothing nested survived. Check whether any cell still holds
#          a dict or a list, and print the result. A neat way:
#            flat_df.map(lambda cell: isinstance(cell, (dict, list))).any()
#          If anything is still nested, fix flatten_response() before moving on.

# YOUR CODE HERE

---
## Part 7 - Clean it, then save it

Same discipline as Assignment 1, and the same reasoning: the raw JSON in `data/raw/` is evidence of what the API sent, and the CSV you are about to write is your interpretation of it. Both go in your repository. Only one of them is ever edited.

Your cleaning function must:

1. **rename columns** to clear lowercase names that carry their units - `life_expectancy_years`, not `value`;
2. **fix the types** - a year to an integer, a date to a real datetime, a measurement to a float;
3. **keep only the columns you need**, especially for option C where thirty columns arrive and perhaps eight matter;
4. **handle missing values explicitly**, remembering that a JSON `null` is a missing measurement and not a zero;
5. **add at least one derived column** that helps answer your question;
6. **sort sensibly** and reset the index;
7. **check plausibility** - print the range of each measure and confirm the values could be real.

In [ ]:
def clean_api_data(df, api_option):
    """Return a cleaned copy of the flattened API data.

    Must not modify the DataFrame passed in.
    """
    df = df.copy()

    # TODO 30: Rename the columns to clear, lowercase, underscore-separated
    #          names that include units where a unit exists.

    # TODO 31: Fix the data types. For option A convert "date" to an integer
    #          year. For option B and the date column of option C, use
    #          pd.to_datetime() - and for option C remember unit="ms".

    # TODO 32: Keep only the columns your analysis needs, in a sensible order.

    # TODO 33: Handle missing values deliberately, and write a comment saying
    #          what you did and why. Do NOT fill a missing measurement with 0.

    # TODO 34: Add at least one derived column. Ideas: the change since the
    #          first year for each country; the percentage change since the
    #          start of the period; a month column extracted from a date; a
    #          magnitude band created with pd.cut().

    # TODO 35: Sort, reset the index with drop=True, and return the DataFrame.

    raise NotImplementedError("Complete TODOs 30 to 35, then delete this line.")


def save_clean_data(df, destination):
    """Save a cleaned DataFrame to CSV and verify it reads back correctly.

    Returns a dict with "path", "rows", "columns", "bytes", "round_trip_ok".
    """
    # TODO 36: Save with index=False, read it straight back, compare the
    #          shapes, and return the dictionary.
    raise NotImplementedError("Complete TODO 36, then delete this line.")

In [ ]:
# TODO 37: Clean your data into clean_df.
# TODO 38: Print the shape before and after cleaning, the new column names, the
#          dtypes, and the remaining missing values.
# TODO 39: Print the range of each numeric column and state in the markdown
#          cell below whether those values are plausible.
# TODO 40: Save clean_df into CLEAN_DIR with a descriptive filename, and print
#          the result of save_clean_data(). Not "clean.csv" - a name that says
#          what the file contains.

# YOUR CODE HERE

### Your answer - cleaning decisions

| Decision | What I did | Why |
|---|---|---|
| Column names | [your answer] | [your answer] |
| Data types | [your answer] | [your answer] |
| Columns dropped | [your answer] | [your answer] |
| Missing values | [your answer] | [your answer] |
| Derived column | [your answer] | [your answer] |

1. Are the ranges of your numeric columns plausible for the real world? Name the largest and smallest value of your main measure and say why each is or is not believable. [your answer]
2. How many missing values did your data contain, and what did you do with them? What would have gone wrong if you had filled them with zero? [your answer]

---
## Part 8 - Analyse

Two functions, each doing one job. Keep them small.

In [ ]:
def summarise_groups(df, group_column, value_column):
    """Summarise the measure for each group.

    Include the count of observations, the first value, the last value, the
    change from first to last, the mean, the minimum, and the maximum.
    Sort by the last value, largest first.

    For option A or B the group is the country or currency. For option C it is
    the month or the magnitude band.
    """
    # TODO 41: Implement with groupby and agg. Remember that "first" and "last"
    #          only mean what you want if the data is sorted in time order,
    #          which your cleaning function should have guaranteed.
    raise NotImplementedError("Complete TODO 41, then delete this line.")


def rank_in_period(df, group_column, value_column, period_column=None,
                   period=None, n=10):
    """Return the top n groups by the measure, optionally within one period.

    If period_column and period are given, filter to that period first.
    """
    # TODO 42: Implement it. Return only the columns needed for a bar chart -
    #          the group and the value - with a reset index.
    raise NotImplementedError("Complete TODO 42, then delete this line.")

In [ ]:
# TODO 43: Run summarise_groups() and display the table.
# TODO 44: Run rank_in_period() for the most recent period in your data.
#          Calculate that period from the data with .max() rather than typing a
#          year, so the cell still works when the source is updated.
# TODO 45: Answer your research question in code, printing the answer as a full
#          sentence with the numbers embedded in it. An f-string does this well:
#            print(f"{name} rose from {first:.1f} to {last:.1f}, a change of {change:.1f}.")

# YOUR CODE HERE

---
## Part 9 - Two charts

The same two charts, and the same standards, as Assignment 1.

1. a **line chart** showing change over time, one line per group;
2. a **bar chart** ranking or comparing groups.

### Every chart must have

- [ ] a title stating the finding, not the mechanics
- [ ] an x-axis label with its unit
- [ ] a y-axis label with its unit
- [ ] a legend whenever more than one series is drawn
- [ ] readable tick labels, rotated if they overlap
- [ ] a source note naming the API and the retrieval date

The tutorial charts in Part 0 meet every one of these. Look back at them for the pattern, including how the source note was added with `fig.text()` and how `ax.bar_label()` put a number on every bar.

In [ ]:
def plot_line_over_time(df, group_column, time_column, value_column,
                        value_label, time_label, title, source_note,
                        save_path=None):
    """Draw a line chart of the measure over time, one line per group.

    Returns the Axes object.
    """
    fig, ax = plt.subplots()

    # TODO 46: Draw one line per group, sorting each group by time first.
    #          A line chart drawn from unsorted data zig-zags backwards and is
    #          one of the easiest ways to publish a wrong chart.

    # TODO 47: Set the title, both axis labels with units, and the legend.

    # TODO 48: Add the source note with fig.text(), add a faint grid, call
    #          fig.tight_layout(), save if save_path was given, and return ax.

    raise NotImplementedError("Complete TODOs 46 to 48, then delete this line.")


def plot_bar_comparison(df, category_column, value_column, value_label,
                        title, source_note, save_path=None):
    """Draw a bar chart comparing the measure across categories.

    Returns the Axes object.
    """
    fig, ax = plt.subplots()

    # TODO 49: Draw the bars, and label each one with ax.bar_label().

    # TODO 50: Set the title and axis labels, rotate the tick labels if they
    #          overlap, put the grid behind the bars with ax.set_axisbelow(True),
    #          add the source note, save if asked, and return ax.

    raise NotImplementedError("Complete TODOs 49 and 50, then delete this line.")

In [ ]:
# TODO 51: Draw your line chart and save it into CHART_DIR.
#          Build the source note from your request_details, so it names the API
#          and the date you actually retrieved the data.
# TODO 52: Draw your bar chart and save it into CHART_DIR.

# YOUR CODE HERE

### Your answer - reading your own charts

1. What does the line chart show? One sentence, at least two numbers. [your answer]
2. What does the bar chart show? One sentence, at least two numbers. [your answer]
3. Why is a line right for the first and a bar right for the second? [your answer]
4. Find something in one of your charts you cannot fully explain - a sudden jump, a gap, an outlier, a line that crosses another. Describe it, and say what extra data you would need to explain it. Every real dataset has one of these, and noticing it is the analyst's job. [your answer]

---
## Part 10 - Pagination, rate limits, and keys

Three things that separate a working script from one that quietly returns incomplete data.

### Pagination

An API will not send you a million records at once. It sends a page and tells you how many pages exist. If you ignore that, you analyse page one and never know the rest existed.

The World Bank response's metadata says exactly this:

```json
{"page": 1, "pages": 3, "per_page": 50, "total": 144}
```

Three pages, and you have one. Your analysis is missing two thirds of the data, and nothing about page one looks wrong.

There are two ways to deal with it: raise `per_page` until everything fits in one response, or loop over pages until `page` equals `pages`. Raising `per_page` is simpler and is what the examples in Part 1 do. Looping is what you need when the total is genuinely large.

### Rate limits

APIs limit how often you may ask. Exceed it and you get HTTP 429. The habits that keep you inside the limit:

- **cache** - save the response and re-read the saved file, rather than requesting again every time you re-run a cell;
- **ask for more per request** - one request for 500 rows beats 500 requests for one row;
- **pause between requests** in a loop, with `time.sleep(1)`;
- **read the terms** - Open-Meteo's free tier is for non-commercial use, and other APIs differ.

Every notebook in this course saves its raw responses to `data/raw/` and re-reads them. That is caching, and it is politeness as well as speed.

### API keys, if you ever use an API that needs one

None of your three options needs a key. When you meet one that does:

**Never type a key into a notebook.** A notebook is a text file, you will commit it to GitHub, and GitHub is public. Scanners find committed keys within minutes, and the consequences range from a revoked key to a bill.

Use an environment variable instead:

```python
import os
api_key = os.environ.get("MY_API_KEY")
if not api_key:
    raise RuntimeError("Set MY_API_KEY before running this notebook.")
```

On Windows, set it for your current terminal session before starting Jupyter or VS Code:

```text
set MY_API_KEY=your-key-here
```

Or in PowerShell:

```text
$env:MY_API_KEY = "your-key-here"
```

The key then exists in your session and never in your files. Also take care not to print it: `print(request_details)` will happily display a key that arrived as a URL parameter, and that output gets saved into your notebook.

In [ ]:
# TODO 53: Investigate completeness. What you do depends on your option.
#
#   Option A, World Bank. Print "page", "pages", "per_page", and "total" from
#     payload[0]. Then confirm the number of records you actually received
#     equals "total". If it does not, you are missing data: either raise
#     per_page and fetch again, or write a loop that requests each page in turn
#     and joins the records together. Either way, print proof that you ended up
#     with all of them.
#
#   Option B, Frankfurter. This API returns the whole range at once, so instead
#     check for GAPS. Count the dates you received, compare that with the number
#     of calendar days in your range, and explain the difference. Weekends and
#     public holidays have no published rate. Print both numbers.
#
#   Option C, USGS. This endpoint caps a response at 20,000 events. Print
#     payload["metadata"]["count"] and compare it with the number of features
#     you received. Then check whether your query came close to the cap, and
#     say what you would do if it had - narrowing the date range or raising
#     minmagnitude are both valid.

# YOUR CODE HERE

### Your answer - completeness

1. How many records did the API say exist, and how many do you have? [your answer]
2. If those numbers differ, what did you do about it? If they match, how would you have known if they had not? [your answer]
3. Describe how a script that ignored pagination could run for months, produce charts every day, and be wrong the whole time without anybody noticing. [your answer]
4. Name two specific things you did in this notebook that reduce the number of requests you send to the API. [your answer]

---
## Part 11 - Manifest, and optional exploration

### Your manifest

Record your endpoint, your parameters, the status code, the retrieval time, the grain, the cleaning steps, and the limitations. Somebody holding this file and your notebook should be able to repeat your work exactly.

In [ ]:
def build_manifest(api, request_details, clean_record, question, cleaning_notes,
                   limitations, grain):
    """Assemble a manifest describing this gathering exercise.

    Structure it with these top-level keys:
      "created_utc", "research_question", "api", "files", "grain",
      "cleaning_steps", "known_limitations", "tools"

    The "api" section must include the endpoint, the documentation URL, the
    parameters actually sent, the status code, the retrieval time, and whether
    a key was required. Look at the tutorial manifest in Part 0 for the shape.
    """
    # TODO 54: Build and return the dictionary.
    #          Do not include an API key, even if your API needed one. A
    #          manifest records WHICH key was used, never its value - name the
    #          environment variable instead.
    raise NotImplementedError("Complete TODO 54, then delete this line.")

In [ ]:
# TODO 55: Write your cleaning steps as a list of clear sentences, one per
#          decision from your Part 7 table.
cleaning_notes = [
    # YOUR ENTRIES HERE
]

# TODO 56: Write at least four honest limitations of your data. Be specific
#          about the claim each one prevents. "The data could be wrong" is not a
#          limitation. "Frankfurter publishes ECB reference rates, which exclude
#          the Ghana cedi, so I cannot say anything about GHS" is.
limitations = [
    # YOUR ENTRIES HERE
]

# TODO 57: Build the manifest, save it to DATA_DIR / "manifest.json" with
#          json.dumps(manifest, indent=2), and print the file contents so the
#          marker can see it in your notebook output.

# YOUR CODE HERE

### Optional - explore further

Not graded. Try one and write a sentence on what it showed you that the required charts did not:

- **a second request** to the same API with different parameters - another indicator, another currency, another magnitude threshold - and compare the two results;
- **a scatter plot** with `ax.scatter()`, if you gathered two numeric measures, to see whether they move together;
- **a horizontal bar chart** with `ax.barh()`, easier to read when category names are long;
- **small multiples** with `plt.subplots(1, 3)`, one panel per group on a shared scale;
- **combining this API with Assignment 1's downloaded file** into one table with `pd.merge()`, which is where gathering starts to become analysis.

In [ ]:
# OPTIONAL - not graded. Explore here.

# YOUR CODE HERE

---
## Part 12 - Your findings, and how to submit

### Write your findings

Full sentences, in the cell below, with a number attached to every claim.

1. **Answer your research question directly.** Three to five sentences, at least three numbers from your analysis.
2. **The most surprising thing you found**, and why it surprised you.
3. **Four limitations of this data.** For each, name the claim it stops you making.
4. **How this API compared with Assignment 1's file download.** Which was more work? Which gave you more control? Which would you choose for a report you had to refresh every month, and why?
5. **One thing that went wrong**, and how you worked it out.

### Then submit

Full instructions, written for Windows with no assumed knowledge, are in [the GitHub submission guide](../topic19d_appendix_submitting_with_github.ipynb). In summary:

1. Create a **public** GitHub repository named `itfy-data-gathering-api`.
2. Copy in this completed notebook, `data/raw/`, `data/clean/`, `charts/`, and `manifest.json`.
3. Write a `README.md` from the template in the guide.
4. Commit with a clear message, then **push**.
5. Open the repository in a private browsing window to prove it is public.
6. Submit the repository link.

The tutorial's own output files may stay in your repository - they show you ran Part 0.

### Before you submit, check every line

- [ ] Every `# TODO` is complete and no `NotImplementedError` remains anywhere
- [ ] The notebook was run from top to bottom, and every output is visible
- [ ] Both charts appear, with titles, axis labels with units, legends, and source notes
- [ ] `data/raw/` holds the saved API responses as received
- [ ] `data/clean/` holds your tidy table
- [ ] `manifest.json` is present and complete
- [ ] The tutorial answers and every "Your answer" cell are filled in - none still says `[your answer]`
- [ ] **No API key, token, or password appears anywhere in the repository**
- [ ] `README.md` is written
- [ ] The repository is public, and the link opens in a private window

### Your findings

**1. Answer to my research question:**

*(Write your answer here.)*

**2. The most surprising thing I found:**

*(Write your answer here.)*

**3. Four limitations of this data:**

1. *(Write your answer here.)*
2. *(Write your answer here.)*
3. *(Write your answer here.)*
4. *(Write your answer here.)*

**4. How the API compared with downloading a file:**

*(Write your answer here.)*

**5. Something that went wrong, and how I worked it out:**

*(Write your answer here.)*

---
## Marking rubric

| Area | What earns the marks | Marks |
|---|---:|---:|
| Tutorial understanding | The five Part 0 questions answered correctly, especially the parallel-arrays and mean-versus-sum questions | 8 |
| Question and API choice | A specific answerable question; parameters genuinely adapted rather than copied from the example | 8 |
| URL building | `build_api_url()` correct, including the empty-parameter case, and the URL checked before sending | 6 |
| Fetch functions | Both functions complete, saving the raw response before parsing, returning payload and details together | 16 |
| Failure handling | All three failures caused and documented; `check_api_payload()` correct and demonstrated on both a good and a bad payload | 14 |
| Structure exploration | `describe_json_structure()` works on nesting without flooding the screen; the response shape described accurately | 8 |
| Flattening | Correct tool for the shape; nothing nested left in any cell | 10 |
| Cleaning | Units in column names; types fixed; missing values handled deliberately and never filled with zero; every decision justified | 14 |
| Saving and raw/clean separation | Descriptive filename, round trip verified, raw responses untouched | 6 |
| Analysis functions | Both correct and used; the question answered in code as a sentence with numbers | 10 |
| Charts | Line and bar correct, fully labelled with units and source notes, and interpreted in words | 12 |
| Pagination and completeness | Completeness actually investigated and evidenced, not asserted | 8 |
| Manifest | Complete, valid JSON, reproducible parameters, no secrets | 6 |
| Findings and limitations | Numbers in every claim; limitations that name the claim they block; the API-versus-download comparison argued | 10 |
| Repository, README, and commits | Public repository, correct structure, complete README, commit messages that describe changes | 8 |
| **Total** | | **144** |

Marks are lost for: leaving `NotImplementedError` in the notebook, parsing before saving the raw response, filling missing measurements with zero, charts without units, asserting completeness without checking it, a manifest containing a secret, and a repository that is not public.

## References

### Your API options
- [World Bank Indicators API - basic call structure](https://datahelpdesk.worldbank.org/knowledgebase/articles/898581-api-basic-call-structures)
- [World Bank Indicators API - developer information](https://datahelpdesk.worldbank.org/knowledgebase/articles/889392)
- [World Bank indicator search](https://data.worldbank.org/indicator)
- [Frankfurter API documentation](https://frankfurter.dev)
- [Frankfurter supported currencies](https://api.frankfurter.dev/v1/currencies)
- [USGS earthquake API documentation](https://earthquake.usgs.gov/fdsnws/event/1/)
- [Open-Meteo historical weather API](https://open-meteo.com/en/docs/historical-weather-api) (used in the tutorial)

### HTTP and JSON
- [Python `urllib.request` documentation](https://docs.python.org/3/library/urllib.request.html)
- [Python `json` documentation](https://docs.python.org/3/library/json.html)
- [`requests` quickstart guide](https://requests.readthedocs.io/en/latest/user/quickstart/)
- [MDN - HTTP response status codes](https://developer.mozilla.org/en-US/docs/Web/HTTP/Status)
- [MDN - an overview of HTTP](https://developer.mozilla.org/en-US/docs/Web/HTTP/Overview)

### pandas and matplotlib
- [`pandas.json_normalize`](https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html)
- [`pandas.DataFrame.from_dict`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.from_dict.html)
- [`pandas.to_datetime`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
- [`pandas.melt` - wide to long](https://pandas.pydata.org/docs/reference/api/pandas.melt.html)
- [pandas group by user guide](https://pandas.pydata.org/docs/user_guide/groupby.html)
- [matplotlib `plot` - line charts](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.plot.html)
- [matplotlib `bar` - bar charts](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.bar.html)

### Course material
- [Topic 19 - Data Gathering lesson](../topic19_data_gathering.ipynb)
- [Appendix - common data sources](../topic19b_appendix_common_data_sources.ipynb)
- [Appendix - reading dataset documentation](../topic19c_appendix_reading_dataset_documentation.ipynb)
- [Appendix - submitting your work with GitHub](../topic19d_appendix_submitting_with_github.ipynb)
- [Assignment 1 - gathering data by downloading a file](topic19_assignment_file_download.ipynb)